In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor, XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import RandomizedSearchCV, train_test_split

### Set variables

In [20]:
races = {'saudi' : 'Saudi Arabian Grand Prix', 'canada' : 'Canadian Grand Prix', 'australia' : 'Australian Grand Prix', 'bahrain' : 'Bahrain Grand Prix', 'spain' : 'Spanish Grand Prix', 'japan' : 'Japanese Grand Prix', 'monaco' : 'Monaco Grand Prix'}
lap_limits = [10, 30]
ordinal_cols = ['Stint', 'Position']  # leave as is
label = ['LapsTilPit']

categorical_cols = ['Driver', 'Team', 'Race', 'TrackStatus', 'Compound']

numerical_cols = ['LapTimeSeconds', 'LapNumber', 'TyreLife',
       'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL',
       'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindSpeed',
       'Speed', 'Throttle', 'RPM', 'Brake', 'nGear', 'Distance', 'DRS']

sector_times = ['Sector1Time', 'Sector2Time', 'Sector3Time',
              'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime']

scaler = StandardScaler()

#### Load cleaned data

In [21]:
for race_key, race_name in races.items():

    # Load clean data
    df = pd.read_csv('data/f1_laps_telemetry_'+race_key+'_clean.csv')

    # Filter out wet tyres
    df = df[~((df['Compound']=='INTERMEDIATE') | (df['Compound']=='WET'))]

    # Encode and standardize features

    # Turn bool values into int
    df['FreshTyre'] = df['FreshTyre'].astype(int)
    df['Rainfall'] = df['Rainfall'].astype(int)

    for lap_num in lap_limits:

        print("Now modeling " + race_name + " performance at Lap " + str(lap_num))

        # Limit dataframe to lap number
        df_copy = df.copy()
        df_current = df_copy[df_copy['LapNumber'] <= lap_num]
        df_next_5 = df_copy[(df_copy['LapNumber'] > lap_num) & (df_copy['LapNumber'] <= (lap_num + 5)]
        
        # Apply cyclical encoding
        df_current['WindDir_sin'] = np.sin(np.deg2rad(df_current['WindDirection']))
        df_current['WindDir_cos'] = np.cos(np.deg2rad(df_current['WindDirection']))
        df_next_5['WindDir_sin'] = np.sin(np.deg2rad(df_next_5['WindDirection']))
        df_next_5['WindDir_cos'] = np.cos(np.deg2rad(df_next_5['WindDirection']))

        df_current = df_current.drop(columns=['WindDirection'])
        df_next_5 = df_next_5.drop(columns=['WindDirection'])

        # Label encode
        encoders = {}

        for col in categorical_cols:
            le = LabelEncoder()
            df_current[col] = le.fit_transform(df_current[col].astype(str))
            df_next_5[col] = le.fit_transform(df_next_5[col].astype(str))
            encoders[col] = le

        # Convert LapTime to seconds
        df_current['LapTimeSeconds'] = pd.to_timedelta(df_current['LapTime']).dt.total_seconds()
        df_next_5['LapTimeSeconds'] = pd.to_timedelta(df_next_5['LapTime']).dt.total_seconds()
        df_current = df_current.drop(columns=['LapTime'])
        df_next_5 = df_next_5.drop(columns=['LapTime'])

        # Convert LapTime to seconds for Sectors
        for col in sector_times:
            df_current[col] = pd.to_timedelta(df_current[col]).dt.total_seconds()
            df_next_5[col] = pd.to_timedelta(df_next_5[col]).dt.total_seconds()

        # Standardize numerical columns
        df_current[numerical_cols] = scaler.fit_transform(df_current[numerical_cols])
        df_next_5[numerical_cols] = scaler.fit_transform(df_next_5[numerical_cols])

        # Split into train, val, and test sets
        df_current_copy = df_current.copy()
        df_next_copy = df_next_5.copy()

        # Binary label: 1 if pitting in next 5 laps (including this lap), else 0
        df_current_copy['PitNext5'] = (df_current_copy['LapsTilPit'] <= 5).astype(int)
        df_next_copy['PitNext5'] = (df_next_copy['LapsTilPit'] <= 5).astype(int)

        # Train, val, test split
        train_df_cl = df_current_copy[df_current_copy['Year'].isin([2022, 2023, 2024])]
        val_df_cl   = df_next_copy[df_next_copy['Year'] == 2024]
        test_df_cl  = df_next_copy[df_next_copy['Year'] == 2025]

        # Split sets into features and target
        feature_cols = [col for col in train_df_cl.columns if col not in ['LapsTilPit', 'Year', 'PitNext5']]

        # Training set
        X_train_cl = train_df_cl[feature_cols]
        y_train_cl = train_df_cl['PitNext5']

        # Validation set
        X_val_cl = val_df_cl[feature_cols]
        y_val_cl = val_df_cl['PitNext5']

        # Test set
        X_test_cl = test_df_cl[feature_cols]
        y_test_cl = test_df_cl['PitNext5']

        print("Train, Validation, and Test Shapes for Race " + race_name + " for Lap " + str(lap_num))
        print(X_train_cl.shape)
        print(y_train_cl.shape)
        print(X_val_cl.shape)
        print(y_val_cl.shape)
        print(X_test_cl.shape)
        print(y_test_cl.shape)

        if (len(X_train_cl) == 0):
            print("Skipping for Lap " + str(lap_num) + " as Years 2022 and 2023 have no data")
            continue
        elif (len(X_val_cl) == 0):
            print("Skipping for Lap " + str(lap_num) + " as Year 2024 has no data")
            continue
        elif (len(X_test_cl) == 0):
            print("Skipping for Lap " + str(lap_num) + " as Year 2025 has no data")
            continue

        # Train XGBoost Classifier model

        # Train
        xgb_base = XGBClassifier()
        xgb_base.fit(X_train_cl, y_train_cl, verbose=0)

        # Predict
        y_val_pred_cl = xgb_base.predict(X_val_cl)

        # Evaluate
        print("Initial XGBoost Classifier Validation Set Performance for " + race_name + " after Lap " + str(lap_num) + ":\n")
        print(confusion_matrix(y_val_cl, y_val_pred_cl))
        print(classification_report(y_val_cl, y_val_pred_cl))

        # Tune XGBoost Classifier model

        # Class imbalance -> calculate ratio of negative to positive class
        ratio = (y_train_cl == 0).sum() / (y_train_cl == 1).sum()

        # Random Search tuning
        param_grid = {
            'n_estimators': [100, 200, 400],
            'max_depth': [4, 6, 8],
            'learning_rate': [0.01, 0.1],
            'subsample': [0.6, 0.8, 1.0],
            'colsample_bytree': [0.6, 0.8, 1.0],
            'scale_pos_weight': [1, ratio*0.5, ratio]
        }

        print("Tuning XGBoost Classifier Performance for " + race_name + " after Lap " + str(lap_num) + ":\n")
        search = RandomizedSearchCV(
            xgb_base,
            param_distributions=param_grid,
            n_iter=72,
            scoring='f1',
            cv=3,
            verbose=0,
            random_state=111
        )
        
        search.fit(X_train_cl, y_train_cl, verbose=0)

        # Validation Set
        best_xgb_cl = search.best_estimator_
        y_val_pred_cl_best = best_xgb_cl.predict(X_val_cl)

        print("Results for Tuned XGBoost Classifier Validation Set Performance for " + race_name + " after Lap " + str(lap_num) + ":\n")
        print(search.best_params_)
        print(confusion_matrix(y_val_cl, y_val_pred_cl_best))
        print(classification_report(y_val_cl, y_val_pred_cl_best))

        # Test Set
        best_xgb_cl = search.best_estimator_
        y_test_pred_cl_best = best_xgb_cl.predict(X_test_cl)

        print("Results for Tuned XGBoost Classifier Test Set Performance for " + race_name + " after Lap " + str(lap_num) + ":\n")
        print(search.best_params_)
        print(confusion_matrix(y_test_cl, y_test_pred_cl_best))
        print(classification_report(y_test_cl, y_test_pred_cl_best))

        # # Train CatBoost Classifier model

        # # Train
        # cat_base = CatBoostClassifier()
        # cat_base.fit(X_train_cl, y_train_cl, verbose=0)

        # # Predict
        # y_val_pred_cat_cl = cat_base.predict(X_val_cl)

        # # Evaluate
        # print("Initial CatBoost Classifier Validation Set Performance for " + race_name + " for Lap " + str(lap_num) + ":\n")
        # print(confusion_matrix(y_val_cl, y_val_pred_cat_cl))
        # print(classification_report(y_val_cl, y_val_pred_cat_cl))

        # # Tune CatBoost Classifier model
        
        # # Random Search tuning
        # param_grid_cl = {
        #     'n_estimators': [100, 200, 400],
        #     'max_depth': [4, 6, 8],
        #     'learning_rate': [0.01, 0.05, 0.1],
        #     'subsample': [0.7, 0.9, 1.0],
        #     'colsample_bylevel': [0.7, 0.9, 1.0],
        #     'scale_pos_weight': [ratio, ratio*0.8, ratio*1.2],
        # }

        # print("Tuning CatBoost Classifier Validation Set Performance for " + race_name + " for Lap " + str(lap_num) + ":\n")
        # search_cat = RandomizedSearchCV(
        #     cat_base,
        #     param_distributions=param_grid_cl,
        #     n_iter=60,
        #     scoring='f1_weighted',
        #     cv=3,
        #     verbose=0,
        #     random_state=111,
        #     n_jobs=-1
        # )
        # search_cat.fit(X_train_cl, y_train_cl, verbose=0)

        # # Validation Set
        # best_cat_cl = search_cat.best_estimator_
        # y_val_pred_cat_cl_best = best_cat_cl.predict(X_val_cl)

        # print("Results for Tuned CatBoost Classifier Validation Set Performance for " + race_name + " for Lap " + str(lap_num) + ":\n")
        # print(search_cat.best_params_)
        # print(confusion_matrix(y_val_cl, y_val_pred_cat_cl_best))
        # print(classification_report(y_val_cl, y_val_pred_cat_cl_best))

        # # Test Set
        # best_cat_cl = search_cat.best_estimator_
        # y_test_pred_cat_cl_best = best_cat_cl.predict(X_test_cl)

        # print("Results for Tuned CatBoost Classifier Validation Test Performance for " + race_name + " for Lap " + str(lap_num) + ":\n")
        # print(search_cat.best_params_)
        # print(confusion_matrix(y_test_cl, y_test_pred_cat_cl_best))
        # print(classification_report(y_test_cl, y_test_pred_cat_cl_best))

Now modeling Saudi Arabian Grand Prix performance at Lap 10
Train, Validation, and Test Shapes for Race Saudi Arabian Grand Prix for Lap 10
(139126, 35)
(139126,)
(62950, 35)
(62950,)
(55129, 35)
(55129,)
Initial XGBoost Classifier Validation Set Performance for Saudi Arabian Grand Prix for Lap 10:

[[33162   564]
 [23540  5684]]
              precision    recall  f1-score   support

           0       0.58      0.98      0.73     33726
           1       0.91      0.19      0.32     29224

    accuracy                           0.62     62950
   macro avg       0.75      0.59      0.53     62950
weighted avg       0.74      0.62      0.54     62950

Tuning XGBoost Classifier Validation Set Performance for Saudi Arabian Grand Prix for Lap 10:

Results for Tuned XGBoost Classifier Validation Set Performance for Saudi Arabian Grand Prix for Lap 10:

{'subsample': 0.8, 'scale_pos_weight': 8.397653086521776, 'n_estimators': 100, 'max_depth': 8, 'learning_rate': 0.01, 'colsample_bytree': 0.

/Users/drewcch/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/drewcch/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/drewcch/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

Initial CatBoost Classifier Validation Set Performance for Saudi Arabian Grand Prix for Lap 10:

[[30956  2770]
 [24260  4964]]
              precision    recall  f1-score   support

           0       0.56      0.92      0.70     33726
           1       0.64      0.17      0.27     29224

    accuracy                           0.57     62950
   macro avg       0.60      0.54      0.48     62950
weighted avg       0.60      0.57      0.50     62950

Tuning CatBoost Classifier Validation Set Performance for Saudi Arabian Grand Prix for Lap 10:

Results for Tuned CatBoost Classifier Validation Set Performance for Saudi Arabian Grand Prix for Lap 10:

{'subsample': 1.0, 'scale_pos_weight': 10.49706635815222, 'n_estimators': 400, 'max_depth': 4, 'learning_rate': 0.01, 'colsample_bylevel': 0.9}
[[32505  1221]
 [27215  2009]]
              precision    recall  f1-score   support

           0       0.54      0.96      0.70     33726
           1       0.62      0.07      0.12     29224

   

/Users/drewcch/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/drewcch/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/drewcch/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Us

Results for Tuned XGBoost Classifier Validation Set Performance for Canadian Grand Prix for Lap 30:

{'subsample': 1.0, 'scale_pos_weight': 4.460676172868884, 'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
[[  0   0]
 [788   0]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00     788.0

    accuracy                           0.00     788.0
   macro avg       0.00      0.00      0.00     788.0
weighted avg       0.00      0.00      0.00     788.0

Results for Tuned XGBoost Classifier Validation Test Performance for Canadian Grand Prix for Lap 30:

{'subsample': 1.0, 'scale_pos_weight': 4.460676172868884, 'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
[[145225   7512]
 [  8874  12082]]
              precision    recall  f1-score   support

           0       0.94      0.95      0.95    152737
           1       

...

Results for Tuned CatBoost Classifier Validation Set Performance for Bahrain Grand Prix for Lap 10:

{'subsample': 0.7, 'scale_pos_weight': 5.4622562366620055, 'n_estimators': 400, 'max_depth': 8, 'learning_rate': 0.05, 'colsample_bylevel': 0.9}
[[53620   735]
 [18128  1484]]
              precision    recall  f1-score   support

           0       0.75      0.99      0.85     54355
           1       0.67      0.08      0.14     19612

    accuracy                           0.74     73967
   macro avg       0.71      0.53      0.49     73967
weighted avg       0.73      0.74      0.66     73967

Results for Tuned CatBoost Classifier Validation Test Performance for Bahrain Grand Prix for Lap 10:

{'subsample': 0.7, 'scale_pos_weight': 5.4622562366620055, 'n_estimators': 400, 'max_depth': 8, 'learning_rate': 0.05, 'colsample_bylevel': 0.9}
[[55285     0]
 [19770     0]]
              precision    recall  f1-score   support

           0       0.74      1.00      0.85     55285
         

/Users/drewcch/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/drewcch/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/drewcch/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

Train, Validation, and Test Shapes for Race Bahrain Grand Prix for Lap 30
(445269, 35)
(445269,)
(221523, 35)
(221523,)
(223889, 35)
(223889,)
Initial XGBoost Classifier Validation Set Performance for Bahrain Grand Prix for Lap 30:

[[168655      0]
 [ 41943  10925]]
              precision    recall  f1-score   support

           0       0.80      1.00      0.89    168655
           1       1.00      0.21      0.34     52868

    accuracy                           0.81    221523
   macro avg       0.90      0.60      0.62    221523
weighted avg       0.85      0.81      0.76    221523

Tuning XGBoost Classifier Validation Set Performance for Bahrain Grand Prix for Lap 30:

Results for Tuned XGBoost Classifier Validation Set Performance for Bahrain Grand Prix for Lap 30:

{'subsample': 1.0, 'scale_pos_weight': 2.6583520248453327, 'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
[[167554   1101]
 [ 40088  12780]]
              precision    recall  f1-

/var/folders/md/k11j6lm14fn29cpdwdfjs85r0000gn/T/ipykernel_85980/3964963562.py:4: DtypeWarning: Columns (4,6,7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/f1_laps_telemetry_'+race_key+'_clean.csv')


Now modeling Monaco Grand Prix performance at Lap 10
Train, Validation, and Test Shapes for Race Monaco Grand Prix for Lap 10
(109698, 35)
(109698,)
(194420, 35)
(194420,)
(65483, 35)
(65483,)
Initial XGBoost Classifier Validation Set Performance for Monaco Grand Prix for Lap 10:

[[ 44165      0]
 [     0 150255]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     44165
           1       1.00      1.00      1.00    150255

    accuracy                           1.00    194420
   macro avg       1.00      1.00      1.00    194420
weighted avg       1.00      1.00      1.00    194420

Tuning XGBoost Classifier Validation Set Performance for Monaco Grand Prix for Lap 10:

Results for Tuned XGBoost Classifier Validation Set Performance for Monaco Grand Prix for Lap 10:

{'subsample': 1.0, 'scale_pos_weight': 35.799060717879904, 'n_estimators': 400, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
[[ 44165      0]
 [121747 